**Student Name:** Yaron Winter

**Assignment Overview**

In this assignment, you will explore two fundamental aspects of modern NLP systems: fine-tuning large language models and understanding the attention mechanism that powers them.

**Learning Goals**

By the end of this assignment, you should be able to:

* Understand when and why to fine-tune language models
* Apply QLoRA for efficient adaptation of large models
* Analyze model outputs and limitations
* Explain and implement the attention mechanism
* Interpret attention patterns and their behavior

**Important Note**

* Do not modify or delete the task structure.
* Complete each task with:
   - Clean, well-organized code
   - Relevant visualizations
   - Clear insights and explanations
* Make sure to answer all required questions.
**Submission requirements:**
* Submit a **fully executed notebook** (all cells must run and outputs should be visible).
* There is no need to attach the training and test datasets as files, but you must present them as DataFrame tables within the notebook.


## Install Required Packages and Dataset

In [40]:
!pip install -q google-colab
!pip install -q bitsandbytes trl
!pip install -q fastapi uvicorn openai
!pip install -q datasets
!pip install -q openai tqdm
!pip install -U trl
!pip install -U torchao
!pip install -q peft

In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline,
    logging,
)
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model,PeftModel
from trl import SFTTrainer
import torch
import json
from datasets import Dataset, load_dataset
import pandas as pd


import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity(logging.CRITICAL)
print("Done.")

Done.


In [4]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cuda


## **Part 1 — QLoRA Fine-Tuning for Level-Adaptive Question Answering (75 points)**

In this part, you will fine-tune pretrained language models to answer questions at different explanation levels:

* Child — simple and intuitive explanation
* Student — clear educational explanation with moderate technical detail
* Expert — precise, technical, and domain-specific explanation

You will use a subset of the Databricks Dolly 15K dataset.

In [5]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:15000]")

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [6]:
def is_good_question(text):
    return any(q in text.lower() for q in [
        "why", "how", "what is", "explain"
    ])

filtered = [
    ex for ex in dataset
    if ex["category"] == "open_qa" and is_good_question(ex["instruction"])
]

In [ ]:
len(filtered), filtered[0]

(1635,
 {'instruction': 'Why can camels survive for long without water?',
  'context': '',
  'response': 'Camels use the fat in their humps to keep them filled with energy and hydration for long periods of time.',
  'category': 'open_qa'})

### **Task 1.1 — Prepare a Custom Dataset (15 points)**

Create your own instruction-tuning dataset based on a subset of databricks-dolly-15k.

For each selected question-answer pair, create versions of the answer adapted to the requested expertise level.

Example format:

    Question: What is gradient descent?
    Expertise level: child
    Answer: Gradient descent is like walking downhill step by step until you reach the lowest point.

You should prepare:

1. Training set for fine-tuning -

   Use the filtered Dolly dataset as your source of questions.

   For each question, create examples in the example format per level.

2. Test set for evaluation -
    
   Create a small test set of 20 question-answer examples.

   The test set must:

  * include examples from all three expertise levels
  * be separate from the training set
  * be manually reviewed by you for quality
  * include answers that are appropriate for the requested expertise level

  Recommendation:

  1. Save the generated dataset as a '.jsonl' file so it can be loaded and reused later.

  2. Decide on the training format according to the model architecture:
    
    - For causal language models, such as `HuggingFaceTB/SmolLM2-360M-Instruct`, use one full text field:

    ```text
    ### Question:
    ...

    ### Expertise level:
    child / student / expert

    ### Answer:
    ...```

    - For seq2seq models, such as google/flan-t5-small, separate the input and target:

    input: Question + expertise level
    target: Answer
   
  3. Before generating the full dataset, test the pipeline on a small batch of examples to verify that:

  * the LLM returns valid JSON,
  * each question receives three expertise-level answers,
  * the saved .jsonl file can be loaded correctly,
  * the format matches the training code.

In [7]:
# Get Nebius API Key
from getpass import getpass
from google.colab import userdata
import os

def get_api_key(name="NEBIUS_API_KEY"):
    api_key = None
    try:
        api_key = os.environ[name]
        print("API Key is taken from the environmental parameters")
    except:
        try:
            api_key = userdata.get(name)
            print("API Key is taken from the google colab user data")
        except:
            try:
                api_key = getpass("Enter API key: ")
                os.environ[name] = api_key
                print("API Key is taken from getpass")
            except:
                raise Exception("API Key for NEBIUS_API_KEY could not be found")

    assert api_key is not None, "API Key is None"
    return api_key
print("Done.")

Done.


In [8]:
import os
from openai import OpenAI

from tqdm import tqdm

# The generation model.
# I use this proprietary model, as in preliminary
# tests it performed much better than the hugging face
# models.
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

QUESTION = "Question"
ANSWER = "Answer"
EXPERTISE = "Expertise"
INSTRUCTION = "instruction"

GENERATION_SYS_PROMPT = """
You are an agent who needs to generate answers for a given questions, according
to the expertises, which are given along the questions.
There are three level of expertises, and this how you should generate your
answer, for each one of them:

Expertise Level: Child
Answer Style: use a simple and intuitive explanation

Expertise Level: Student
Answer Style: use clear educational explanation with moderate technical detail

Expertise Level: Expert
Answer Style: use precise, technical, and domain-specific explanation

These would be the responses for the next example question,
based on the requested expertise level:

Question Example: Why can camels survive for long without water?

Child Response: Camels use the fat in their humps to keep them filled with energy
                and hydration for long periods of time.

Student Response: Camels can survive long periods without water because they have
                  adaptations that minimize water loss and maximize efficiency.
                  Their kidneys and intestines conserve water by producing highly
                  concentrated urine and dry feces, and they can tolerate large
                  fluctuations in body temperature to reduce sweating.

Expert Response: Camels exhibit a suite of physiological adaptations that enable
                extreme dehydration tolerance, including highly efficient renal
                concentrating ability and reduced evaporative water loss via adaptive
                heterothermy. Their erythrocytes are oval and highly deformable, allowing
                circulation under increased blood viscosity during dehydration. Fat
                stored in the hump can be oxidized to yield metabolic water, contributing marginally to hydration
"""

def build_prompt(question: str, expertise: str) -> str:
        return f"""
        Answer the following question according to the requested expertise level.

        Question: {question}
        Expertise level: {expertise}

        Please limit your response to no more than 4 sentences,
        and retrieve it as a string.
"""


client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=get_api_key()
)

def generate_response(question: str, expertise: str, model_name=MODEL_NAME) -> dict:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": GENERATION_SYS_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": build_prompt(question, expertise)
                    }
                ]
            }
        ]
    )
    return json.loads(response.to_json())

def generate_dataset(questions: list) -> pd.DataFrame:
    ds = {QUESTION: [], EXPERTISE: [], ANSWER: []}
    expertises = ["child", "student", "expert"]

    for question in tqdm(questions):
        for expertise in expertises:
          answer = generate_response(question=question, expertise=expertise)

          ds[QUESTION].append(question)
          ds[EXPERTISE].append(expertise)
          ds[ANSWER].append(answer["choices"][0]["message"]["content"])

    return pd.DataFrame(ds)

print("Done.")

API Key is taken from the google colab user data
Done.


In [ ]:
# Test the generation pipeline.
questions = [x[INSTRUCTION] for  x in filtered[:3]]
print(questions)
df = generate_dataset(questions=questions)
df.head(10)

['Why can camels survive for long without water?', 'What is a polygon?', 'What is a verb?']


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:45<00:00, 15.29s/it]


,Question,Expertise,Answer
0,Why can camels survive for long without water?,child,"""Camels use the fat in their humps to keep the..."
1,Why can camels survive for long without water?,student,Camels can survive long periods without water ...
2,Why can camels survive for long without water?,expert,Camels exhibit a suite of physiological adapta...
3,What is a polygon?,child,"""A polygon is a shape that has lots of sides! ..."
4,What is a polygon?,student,"""A polygon is a two-dimensional shape with at ..."
5,What is a polygon?,expert,"""A polygon is a two-dimensional geometric figu..."
6,What is a verb?,child,"""A verb is a word that tells us what someone o..."
7,What is a verb?,student,"""A verb is a word that expresses action, occur..."
8,What is a verb?,expert,"""Verbally, a verb is a word that expresses act..."


**Conclusions & Observations from the initial tests:**

Overall, the results seem reasonable.
I must admit, though, that it is very hard to distnguish between
student answers to an expert answers.
In fact, it would be difficult for me to explain how such differences would look
like, or to distinguish between them.

In [ ]:
# Generate the datasets.
# I start with 250 training questions.
# Notice that each question will be ganswered
# in three styles, so it induces a train set
# of 750 entries.
print(f"Total filtered dataset size: {len(filtered)}")

NUM_TRAIN_QUESTIONS = 250
train_questions = [x[INSTRUCTION] for  x in filtered[:NUM_TRAIN_QUESTIONS]]

# The test questions are seperated from the training questions, of course.
# Generate more test questions than needed, and later I will select manually
# the most appropraite ones.
test_questions = [x[INSTRUCTION] for  x in filtered[NUM_TRAIN_QUESTIONS: NUM_TRAIN_QUESTIONS + 50]]

train_df = generate_dataset(train_questions)
test_df = generate_dataset(test_questions)

train_df.to_json("dataset_train.jsonl", orient='records', lines=True)
test_df.to_json("dataset_test.jsonl", orient='records', lines=True)
print("Done.")

Total filtered dataset size: 1635


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [08:47<00:00, 10.55s/it]

Done.


### **Task 1.2 — Fine-Tune Models with QLoRA (10 points)**

Fine-tune the models using QLoRA, meaning:

1. Load the base model in 4-bit quantization
2. Add LoRA adapters
3. Train only the adapter parameters

You should repeat the fine-tuning process for both models:

1. HuggingFaceTB/SmolLM2-360M-Instruct
2. google/flan-t5-small

Pay attention each model requires the dataset to be formated diffrently.

In [11]:
# Fine tune the google/flan-t5-small LLM

# Create the dataloader for the fine tuning process.
dataset = load_dataset("json", data_files="dataset_train.jsonl")

def format_item(item: dict) -> dict:
    input_text = f"Question: {item['Question']} Expertise: {item['Expertise']}"
    target_text = item["Answer"]

    return {
        "input_text": input_text,
        "target_text": target_text
    }

dataset = dataset.map(format_item)
print(type(dataset))

<class 'datasets.dataset_dict.DatasetDict'>


In [12]:
# Tokenize the dataset
MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

INPUT_IDS= "input_ids"
LABELS = "labels"
ATTENTION_MASK = "attention_mask"

max_input_length = 256
max_target_length = 128

def tokenize(item: dict):
    model_input = tokenizer(
        item["input_text"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        item["target_text"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_input[LABELS] = labels[INPUT_IDS]
    return model_input

tokenized_dataset = dataset.map(tokenize, batched=False)
tokenized_dataset.set_format(type="torch")
print(type(tokenized_dataset))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

<class 'datasets.dataset_dict.DatasetDict'>


In [13]:
# Load the model in 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Add LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 2,555,904 || all params: 79,517,056 || trainable%: 3.2143


In [14]:
# Create the data loader
from torch.utils.data import DataLoader

train_dataset = tokenized_dataset["train"]

train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)
print(type(train_dataloader))

# define the training procedure.
EARLY_STOP_PATIENCE = 2
def train_google_flan(net):
    optimizer = torch.optim.Adam(net.parameters(), lr=2e-4)
    net = net.to(device)
    net.train()

    num_epochs = 20
    best_loss = None
    num_no_imp = 0
    for epoch in range(num_epochs):
        total_loss = 0.0
        for batch in tqdm(train_dataloader):
            batch = {
                INPUT_IDS: batch[INPUT_IDS].to(device),
                LABELS: batch[LABELS].to(device),
                ATTENTION_MASK: batch[ATTENTION_MASK].to(device)
            }

            outputs = net(**batch)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        if (best_loss is None) or (total_loss <= best_loss):
            num_no_imp = 0
            best_loss = total_loss
        else:
            num_no_imp += 1

        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")
        if num_no_imp >= EARLY_STOP_PATIENCE:
            print("Early Stop!")
            break

    # Save the LoRA adapters.
    model.save_pretrained("flan-t5-small-lora")
    tokenizer.save_pretrained("flan-t5-small-lora")

<class 'torch.utils.data.dataloader.DataLoader'>


In [ ]:
# Fine tune the model.
train_google_flan(net=model)

100%|██████████| 94/94 [00:23<00:00,  3.92it/s]


Epoch 1, Loss: 906.9301


100%|██████████| 94/94 [00:30<00:00,  3.09it/s]


Epoch 2, Loss: 778.2249


100%|██████████| 94/94 [00:23<00:00,  3.93it/s]


...

Epoch 18, Loss: 681.1831


100%|██████████| 94/94 [00:23<00:00,  3.97it/s]


Epoch 19, Loss: 680.7697


100%|██████████| 94/94 [00:23<00:00,  3.98it/s]


Epoch 20, Loss: 680.2454


In [15]:
# Fine tune HuggingFaceTB/SmolLM2-360M-Instruct model

# Generate the dataset for the fine tuning.
dataset = load_dataset("json", data_files="dataset_train.jsonl")
def format_casual_lm_item(item: str) -> str:
    text = f"""Question: {item[QUESTION]}
                Expertise: {item[EXPERTISE]}


                Answer: {item[ANSWER]}"""

    return {"text": text}


dataset = dataset.map(format_casual_lm_item)

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [16]:
# Tokenize the dataset
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

max_length = 256

def tokenize_casual_llm(item: dict):
    tokens = tokenizer(
        item["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_casual_llm)

# Remove text columns
tokenized_dataset = tokenized_dataset.remove_columns(
    dataset["train"].column_names
)

tokenized_dataset.set_format("torch")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [17]:
# Allocate the base model.
# Notice that the bnb_config remains the same
# as it was set for the sequence-2-sequence model.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [18]:
# Prepare the model for k-bit training.
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# Add the LoRA adapters.
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,276,800 || all params: 365,097,920 || trainable%: 0.8975


In [19]:
# Set the data loader.
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

train_dataloader = DataLoader(
    tokenized_dataset["train"],
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

In [20]:
# Define the training procedure.

EARLY_STOP_PATIENCE = 2
def train_casual_llm(net):
    optimizer = torch.optim.Adam(net.parameters(), lr=2e-4)
    net.train()

    num_epochs = 7
    num_not_imp = 0
    best_loss = None
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in tqdm(train_dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = net(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        if (best_loss is None) or (total_loss < best_loss):
            num_no_imp = 0
            best_loss = total_loss
        else:
            num_no_imp += 1

        print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")
        if num_no_imp >= EARLY_STOP_PATIENCE:
            print("Early Stop!")
            break

In [ ]:
# Train the model
train_casual_llm(net=model)

# Save the trained model
model.save_pretrained("smollm2-lora")
tokenizer.save_pretrained("smollm2-lora")

100%|██████████| 94/94 [01:29<00:00,  1.04it/s]


Epoch 1, Loss: 131.3857


100%|██████████| 94/94 [01:28<00:00,  1.06it/s]


Epoch 2, Loss: 125.8954


100%|██████████| 94/94 [01:28<00:00,  1.06it/s]


...

100%|██████████| 94/94 [01:28<00:00,  1.06it/s]


Epoch 6, Loss: 99.9320


100%|██████████| 94/94 [01:28<00:00,  1.06it/s]


Epoch 7, Loss: 92.7313


('smollm2-lora/tokenizer_config.json',
 'smollm2-lora/chat_template.jinja',
 'smollm2-lora/tokenizer.json')

### **Task 1.3 — Evaluation (20 points)**

In [55]:
print("The Test Set:\n\n")
import pandas as pd

df = pd.read_json("test_set.jsonl", lines=True)
df.head(10)

The Test Set:




,Question,Expertise,Answer,Original_Answer
0,What is a knowledge base?,child,"""A knowledge base is like a big library in a c...",A knowledge base is a set of articles composed...
1,What is a knowledge base?,expert,"""A knowledge base is a computer system designe...",A knowledge base is a set of articles composed...
2,How do sailplanes (gliders) stay aloft?,child,Sailplanes stay aloft because they use rising ...,Gliders are usually towed to altitude by a mot...
3,How do sailplanes (gliders) stay aloft?,student,Sailplanes (gliders) stay aloft by converting ...,Gliders are usually towed to altitude by a mot...
4,What is locus (in genomics)?,expert,"""Locus in genomics refers to a specific locati...","In genetics and genomics, a locus is a specifi..."
5,Why do players smear black under their eyes?,student,"Players smear black under their eyes, also kno...",Black eye smear reduces the blinding sunlight ...
6,How fast can an ostrich run?,child,"""Ostriches are very fast birds! They can run u...",An ostrich can run up to 56 mph (90 km/h). It ...
7,How fast can an ostrich run?,expert,"""An ostrich's running speed can reach up to 70...",An ostrich can run up to 56 mph (90 km/h). It ...
8,How old was Mozart when he first performed?,student,Mozart made his public debut at the age of fiv...,He was six years old when he first performed i...
9,What is behavioural economics?,child,Behavioural economics is a way of understandin...,A method of economic analysis that applies psy...


In [26]:
# Start by uploading and unzipping the trained models.
import zipfile

zip_path = "flan-t5-small-lora.zip"
extract_path = "."

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

zip_path = "smollm2-lora.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [56]:
# Allocate the models.

# Allocate the seq-2seq fine tuned model
base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", quantization_config=bnb_config, device_map="auto")
s2s_fine_tuned_model = PeftModel.from_pretrained(base_model, "content/flan-t5-small-lora")

# Allocate the seq-2seq base model
s2s_base_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
s2s_tokenizer = AutoTokenizer.from_pretrained("content/flan-t5-small-lora")

# Allocate the casual lm fine tuned model
base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct", quantization_config=bnb_config, device_map="auto")
casual_lm_finetuned_model = PeftModel.from_pretrained(base_model, "content/smollm2-lora")

# Allocate the casual lm base model
casual_lm_base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
casual_lm_tokenizer = AutoTokenizer.from_pretrained("content/smollm2-lora")

s2s_base_model = s2s_base_model.to(device).eval()
s2s_fine_tuned_model = s2s_fine_tuned_model.to(device).eval()
casual_lm_base_model = casual_lm_base_model.to(device).eval()
casual_lm_finetuned_model = casual_lm_finetuned_model.to(device).eval()

def convert_response(response: str) -> str:
    ind = response.find("Answer:")
    words = response[ind + 7:].strip().split()
    return " ".join(words)

def indentity_func(response: str) -> str:
  return response

def generate_answer(model, tokenizer, question, expertise, conversion):
    prompt = f"""
    Answer the question according to the requested expertise level.

    Question: {question}
    Expertise: {expertise}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=1.7)
    return conversion(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [51]:
answer = generate_answer(s2s_fine_tuned_model, s2s_tokenizer, "What is a knowledge base?", "child", indentity_func)
print(f"original answer: {answer}")

original answer: "A a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a aas a


Evaluate both fine-tuned models on your 20-example test set.

For each model, compare:

* outputs before fine-tuning
* outputs after fine-tuning
* whether the answer matches the requested expertise level
* whether the answer is clear and relevant
* whether the answer stays faithful to the question

Evaluate using this table:

|Question|Level|Base Output|Fine-tuned Output|Expected Answer|Better after FT?|Notes|
|-------|-------|-------|-------|-------|-------|-------|


In [61]:
LEVEL = "level"
BASE = "Base_Output"
FINE_TUNED = "Fine_Tuned_Output"
EXPECTED = "Expected_Answer"
IS_BETTER = "Better_After_FT?"
COMMENT = "Notes"
ORIGINAL_ANS = "Original_Answer"

def evaluate_models(base_model, fine_tuned_model, tokenizer, test_df, conversion) -> pd.DataFrame:
    results = {
        QUESTION: [],
        LEVEL: [],
        BASE: [],
        FINE_TUNED: [],
        EXPECTED: [],
        IS_BETTER: [],
        COMMENT: [],
    }

    for i in tqdm(range(len(test_df))):
        question = test_df.loc[i, QUESTION]
        expertise = test_df.loc[i, EXPERTISE]

        results[QUESTION].append(question)
        results[LEVEL].append(expertise)
        results[BASE].append(generate_answer(base_model, tokenizer, question, expertise, conversion))
        results[FINE_TUNED].append(generate_answer(fine_tuned_model, tokenizer, question, expertise, conversion))
        results[EXPECTED].append(test_df.loc[i, ORIGINAL_ANS])
        results[IS_BETTER].append("")
        results[COMMENT].append("")

    return pd.DataFrame(results)

In [64]:
# Eveluate the seq-2-seq models.
res_df = evaluate_models(s2s_base_model, s2s_fine_tuned_model, s2s_tokenizer, df, indentity_func)
res_df.to_json("results_s2s.jsonl", orient='records', lines=True)

100%|██████████| 20/20 [01:34<00:00,  4.72s/it]


In [65]:
# Eveluate the casual lm models.
res_df = evaluate_models(casual_lm_base_model, casual_lm_finetuned_model, casual_lm_tokenizer, df, convert_response)
res_df.to_json("results_casual_lm.jsonl", orient='records', lines=True)

100%|██████████| 20/20 [05:32<00:00, 16.63s/it]


In [68]:
# The results of the google flan model, after analysis
res_df = pd.read_json("results_s2s.jsonl", lines=True)
res_df.head(15)

,Question,level,Base_Output,Fine_Tuned_Output,Expected_Answer,Better_After_FT?,Notes
0,What is a knowledge base?,child,a child,"""A a aas a aas a aas a aas a aas a aas a aas a...",A knowledge base is a set of articles composed...,No!,Fine Tuning crashed completely
1,What is a knowledge base?,expert,a knowledge base,"""A a aas a aas a aas a aas a aas a aas a aas a...",A knowledge base is a set of articles composed...,No!,Fine Tuning crashed completely
2,How do sailplanes (gliders) stay aloft?,child,a sailor,"""Sasas a asassas a asassas a asassa a asassas ...",Gliders are usually towed to altitude by a mot...,No!,Fine Tuning crashed completely
3,How do sailplanes (gliders) stay aloft?,student,a student,"""Sasas a asassas a asassas a asassas a asassa ...",Gliders are usually towed to altitude by a mot...,No!,Fine Tuning crashed completely
4,What is locus (in genomics)?,expert,locus,"""As a a aas a aas a aas a aas a aas a aas a aa...","In genetics and genomics, a locus is a specifi...",No!,Fine Tuning crashed completely
5,Why do players smear black under their eyes?,student,smear black under their eyes,"""A aas a aas a aas a aas a aas a aas a aas a a...",Black eye smear reduces the blinding sunlight ...,No!,Fine Tuning crashed completely
6,How fast can an ostrich run?,child,a frog,"""A aas a aas a aas a aas a aas a aas a aas a a...",An ostrich can run up to 56 mph (90 km/h). It ...,No!,Fine Tuning crashed completely
7,How fast can an ostrich run?,expert,a meter,"""A aas a aas a aas a aas a aas a aas a aas a a...",An ostrich can run up to 56 mph (90 km/h). It ...,No!,Fine Tuning crashed completely
8,How old was Mozart when he first performed?,student,18,"""As a a aas, a aas a aas a aas a aas a aas a a...",He was six years old when he first performed i...,No!,Fine Tuning crashed completely
9,What is behavioural economics?,child,behavioural economics,"""A adapaas is a a ad ad a ad a ad a aas a aas ...",A method of economic analysis that applies psy...,No!,Fine Tuning crashed completely


In [69]:
# The results of the SmolLM2 model, after analysis
res_df = pd.read_json("results_casual_lm.jsonl", lines=True)
res_df.head(15)

,Question,level,Base_Output,Fine_Tuned_Output,Expected_Answer,Better_After_FT?,Notes
0,What is a knowledge base?,child,1. A knowledge base is a database that stores ...,A knowledge base is like a big library where c...,A knowledge base is a set of articles composed...,Yes,More like an child
1,What is a knowledge base?,expert,1. A knowledge base is a database that stores ...,"""A knowledge base is a repository of informati...",A knowledge base is a set of articles composed...,Yes,More like an expert
2,How do sailplanes (gliders) stay aloft?,child,1. They use wind. 2. They use wings. 3. They u...,"Sailplanes, also called gliders, stay aloft by...",Gliders are usually towed to altitude by a mot...,Yes,"Better formatted, a bit more like child"
3,How do sailplanes (gliders) stay aloft?,student,1. Wind resistance: Sailplanes use wind resist...,Sailplanes (gliders) use a combination of lift...,Gliders are usually towed to altitude by a mot...,Yes,More like a student
4,What is locus (in genomics)?,expert,1. The locus is a specific location on a chrom...,"""Locus in genomics refers to the location or p...","In genetics and genomics, a locus is a specifi...",Yes,More coherently and like an expert
5,Why do players smear black under their eyes?,student,1. To make their eyes look more attractive. 2....,"""Players often apply black eyeliner to create ...",Black eye smear reduces the blinding sunlight ...,Not Really,"Tone more like a student, but both were wrong"
6,How fast can an ostrich run?,child,1. Ostrich can run at a speed of 40-50 km/h. Q...,"""An ostrich can run really fast! It can run as...",An ostrich can run up to 56 mph (90 km/h). It ...,Yes,More precise and like a child
7,How fast can an ostrich run?,expert,1. Ostrich can run at a speed of 40-50 km/h. Q...,"""An ostrich's running speed can vary depending...",An ostrich can run up to 56 mph (90 km/h). It ...,Not Really,"Tone like an expert, but wrong"
8,How old was Mozart when he first performed?,student,1. Mozart was 12 years old when he first perfo...,10 years old. Question: What is the capital of...,He was six years old when he first performed i...,Not Really,Both hallucinated and failed on format
9,What is behavioural economics?,child,1. The study of human behaviour in the context...,"""Behavioral economics is a way of thinking abo...",A method of economic analysis that applies psy...,Yes,"More precise, formatted, and like a child"


### **Task 1.4 — Model Comparison and Discussion (15 points)**

Compare the performance of the two models:

* Which model adapted better to the expertise levels?
* Which model produced clearer answers?
* Which model followed the requested format better?
* Did one model hallucinate more than the other?

Explain any differences you observe.

In your discussion, consider that:

* SmolLM2-360M-Instruct is a decoder-only instruction model
* flan-t5-small is an encoder-decoder instruction model
* Different architectures may behave differently on instruction-following and text generation tasks



*   The ***SmolLM2-360M-Instruct*** model performed much better than ***flan-t5-small***, both before and after the fine tuning
*   While ***SmolLM2-360M-Instruct*** adapted very well to the task, following the fine tuning, the performance of the tuned ***flan-t5-small*** model completely crashed - all the answers of the tuned ***flan-t5-small*** model were pure garbage


*   ***SmolLM2-360M-Instruct***  adapted very well to the task, following its fine tuning:
    * the style of its answers matched the requested expert in most cases
    * the format was correct with almost no gliches (e.g. repetitions, repeat the question, etc.)
    * it even admitted not to knowing the answer in one case
    * it was pretty precise in most cases
    * it had less hallucinations, and answered right to the point more often
* The performance of ***flan-t5-small*** model was flan-t5-small poor:
    * The base model did not answer the questions, but rather only repeated a few words of them, usually randomly
    * The fine tuned model generated just sequences of letters combinations (usually 'a's) - a total crash
    * I made some efforts to resolve it
        * Consulted the chats massively
        * tried flan-t5-large model, whose base version gave much better results
    * But further digging in this matter is far beyond the scope of this assignment, so I moved on...
* The differences in performance  between these two models may stem from both the architectures and the sizes:
    * The fact that most (all?) generative LLM models consist of decoder only architecture hint that this architecture may be inherently better for generative tasks, such as we had in this assignment
    * Besides, ***SmolLM2-360M-Instruct*** is much bigger than ***flan-t5-small***, which is also a crucial factor

### **Task 1.5 - Conceptual Questions (15 points)**

Answer the following questions:

1. What changed after fine-tuning?
   
   Discuss whether the model became better at adapting its answer to the requested expertise level.
2. Why is QLoRA more memory efficient?
   
   Explain the role of 4-bit quantization and LoRA adapters.
3. What happens if you increase the LoRA rank?
   
   Discuss the tradeoff between model capacity, memory usage, and overfitting risk.
4. Why use LoRA / QLoRA instead of prompt engineering?

      Discuss:

      * In what cases prompt engineering is sufficient
      * When fine-tuning becomes necessary
      * What advantages QLoRA provides over prompting
      * What are the trade-offs (cost, flexibility, control)


*   What changed after fine-tuning?

    * ***flan-t5-small*** completely crashed, as detailed above
    * ***SmolLM2-360M-Instruct*** improved significantly
        * adapted its answers very well to the requested expertise level
        * were more precise and to the point
*   Why is QLoRA more memory efficient?
    * QLoRA is used to reduce the storage of each matrix entry from 16 bits (in float16) or 32 bits (float32) to 4 bits
    * This reduces the overall storage of the matrices by factor of 4 or 8
    * The LoRA adapters reduce the number of trained parameters by 2 to 4  magnitutes
        * it is done by replacding large matrices by two much lower rank matrices, whose multiplication produces the same effect as the large matrix multiplication
* What happens if you increase the LoRA rank?
    * Increasing LoRA induces increase of the learned parameters number, of course
    * But it also may induce weaker gradients signals, due to the ranking scalling (i.e. *alpha*/*rank*)
* Why use LoRA / QLoRA instead of prompt engineering?
    * prompt engineering would be better, when it is sufficient for obtaining satisfactory performance, for example:
        * When we need to deduce some conclusion or extract some information from a given text
        * When by prompting alobe we can obtain satisfactory performance
    * Fine Tuning may be used:
        * When obtaining consistent style for our responses is crucial
        * When we have sufficient and good training data
        * For high traffic, when using our own trained model may spare costs significantly
* What advantages QLoRA provides over prompting?
    * QLoRA is used for improving fine tuning of large models
        * it requires less memory and computation resources
        * it enables training and mainting many fine tuned models, which are based on a single base model
        * it helps to prevent catastrophic forgetting
        * it has a good and comfortable framework supprt - easy for implementation
    * So in cases where fine tuning are better than prompt engineering - QLoRA is useful

  * What are the trade-offs (cost, flexibility, control)?
      * Fine tuning procedure is always a complicated and costly task
          * the need for training data
          * computation resources
          * human efforts
          * final satisfactory and consistent results are not guaranteed
      * So as long as the circumstances do not requires fine tuning, prompt engineering of RAG will also be much less costly, very flexible and quick, and more appropriate for responding effective on business requirements



## **Part 2 — Understanding Attention (25 points)**

Goal:

Build intuition for how attention works.

Task:

You will implement a simple attention mechanism from scratch (PyTorch) and visualize its behavior.

### **Task 2.1 Implement Scaled Dot-Product Attention (5 points)**

Given:

* Query (Q)
* Key (K)
* Value (V)

Compute:

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implements Scaled Dot-Product Attention.

    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: (batch, heads, seq_len_q, seq_len_k) or None

    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """


    #  Get dimension for scaling
    d_k = Q.size(-1)


    # Compute attention scores

    scores = # TODO: compute dot-product between Q and K^T


    # Scale the scores

    # TODO: divide scores by sqrt(d_k)


    #  Apply mask (if given)

    if mask is not None:
        # TODO: mask out invalid positions (set to -inf)
        pass


    # Softmax to get attention

    attention_weights = # TODO: apply softmax over last dimension


    # Compute weighted sum

    output = #TODO: multiply attention_weights with V

    return output, attention_weights

### **Task 2.2 Visualize Attention (5 points)**


Plot attention weights as a heatmap for the given sentence


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import math



sentence = "the cat sat on the mat"


tokens = # TODO: split sentence into tokens

seq_len = # TODO: compute sequence length



# Create dummy embeddings

d_model = 16


embeddings = # TODO: initialize random embeddings of shape (seq_len, d_model)

# Treat embeddings as Q, K, V
Q = embeddings
K = embeddings
V = embeddings



# Compute Scaled Dot-Product Attention

attention_weights = # TODO: compute attention weights (use function from above)



# Plot attention heatmap

plt.figure(figsize=(8, 6))

# TODO: plot heatmap using seaborn
# - use attention_weights
# - set xticklabels and yticklabels to tokens
# - choose a colormap (e.g., "viridis")

plt.title("Attention Weights Heatmap")

# TODO: label axes
# plt.xlabel(...)
# plt.ylabel(...)

# TODO: rotate ticks if needed

plt.show()



### **Task 2.3 Experiments (15 points)**

**Experiment 1 — Change One Word:**

1. Use a simple sentence, for example:
    the cat sat on the mat
2. Compute and plot the attention weights as a heatmap.
3. Change one word in the sentence, for example:
    the dog sat on the mat
4. Recompute and plot the attention heatmap.
5. Compare the two heatmaps and explain whether the attention pattern changed.

**Experiment 2 — Compare Different Attention Heads:**

Repeat the attention visualization using at least two different attention heads.

For each head, create separate projection matrices:

$$W_Q, W_K, W_V$$

Use them to compute:
$$Q=XW_Q, K=XW_K, V=XW_V$$

Then compute and plot the attention weights for each head.

**Questions**

Answer briefly:

1. Did changing one word affect the attention weights? Why or why not?
2. Do different attention heads focus on different tokens?
3. Why might multi-head attention be useful in Transformer models?